<a href="https://colab.research.google.com/github/jamalinu/amazigh-nlp-spacy/blob/main/Tarifit_PBC_Corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building the First Tarifit TTS Voice: A Phonetically Balanced Corpus for Riffian Berber

# 🔊 Phonetically Balanced Corpus — Tarifit (Riffian Berber)
## TTS Data Preparation · IPA Transcription · Phoneme Coverage

---

Este notebook construye un corpus de 50 frases en Tarifit
diseñadas para entrenar un sistema Text-To-Speech (TTS).

**Autor:** jamalinu
**Lengua:** Tarifit (Bereber del Rif) — ISO 639-3: rif

In [ ]:
# Esta es una celda de código Python
# Las líneas que empiezan con # son comentarios
# Python no las ejecuta — sirven para explicar el código

print("Notebook Tarifit PBC listo")
print("Empezamos a construir el corpus")

In [ ]:
# ============================================================
# STEP 1 — Import libraries
# 'import' le dice a Python: "quiero usar esta librería"
# 'as pd' significa: la llamaré 'pd' para abreviar
# ============================================================

import pandas as pd
# pandas sirve para crear y manejar tablas de datos
# la abreviatura 'pd' es la convención universal

import matplotlib.pyplot as plt
# matplotlib dibuja gráficas
# 'pyplot' es el módulo específico para dibujar
# lo llamamos 'plt' por convención

import seaborn as sns
# seaborn hace gráficas más bonitas sobre matplotlib

import re
# 're' es la librería de expresiones regulares
# la usaremos para buscar patrones en texto IPA

from collections import Counter
# Counter cuenta cuántas veces aparece cada elemento
# ejemplo: Counter(['a','b','a']) → {'a':2, 'b':1}

# ── Confirmación ──────────────────────────────────────────
print("Libraries imported:")
print(f"  pandas     version: {pd.__version__}")
print(f"  matplotlib version: {plt.matplotlib.__version__}")
print("  re         (built-in)")
print("  Counter    (built-in)")
print()
print("Ready to build the Tarifit PBC Corpus!")

# ---

# ### ¿Qué significa cada símbolo?

# | Símbolo | Nombre | Qué hace |
# |---------|--------|---------|
# | `#` | Comentario | Python lo ignora, es para ti |
# | `import` | Importar | Carga una librería |
# | `as` | Alias | Le da un nombre corto |
# | `from` | Desde | Importa solo una parte |
# | `print()` | Imprimir | Muestra texto en pantalla |
# | `f"..."` | f-string | Texto que incluye variables |

# ---

# ### Cuando ejecutes la celda deberías ver:
# ```
# Libraries imported:
#   pandas     version: 2.x.x
#   matplotlib version: 3.x.x
#   re         (built-in)
#   Counter    (built-in)

# Ready to build the Tarifit PBC Corpus!
# ```

In [ ]:
# ============================================================
# STEP 2 — Tarifit Phoneme Inventory
#
# Un diccionario de diccionarios:
# La CLAVE exterior es la categoría (vowels, stops...)
# La CLAVE interior es el símbolo IPA
# El VALOR es la descripción del sonido
# ============================================================

PHONEME_INVENTORY = {

    # VOWELS — Tarifit tiene solo 3 vocales básicas
    # (el árabe y el bereber tienen sistemas vocálicos reducidos)
    'vowels': {
        'a': 'open central unrounded',
        'i': 'close front unrounded',
        'u': 'close back rounded',
    },

    # STOPS — consonantes oclusivas
    # Las que llevan ˁ son ENFÁTICAS (faringalizadas)
    # Son únicas del árabe y bereber — críticas para TTS
    'stops': {
        'b':  'voiced bilabial',
        'd':  'voiced dental',
        'dˁ': 'voiced emphatic dental',
        'g':  'voiced velar',
        'k':  'voiceless velar',
        'q':  'voiceless uvular',
        't':  'voiceless dental',
        'tˁ': 'voiceless emphatic dental',
    },

    # FRICATIVES — consonantes fricativas
    # 'ɣ' es el sonido de 'gh' en Tarifit (ghur, ffigh)
    # 'ħ' es el sonido de 'h' fuerte (ruḥ, yelha)
    'fricatives': {
        'f':  'voiceless labiodental',
        's':  'voiceless alveolar',
        'sˁ': 'voiceless emphatic alveolar',
        'z':  'voiced alveolar',
        'zˁ': 'voiced emphatic alveolar',
        'ʃ':  'voiceless postalveolar',
        'x':  'voiceless velar',
        'ɣ':  'voiced velar fricative',
        'h':  'voiceless glottal',
        'ħ':  'voiceless pharyngeal',
        'ʕ':  'voiced pharyngeal',
    },

    # AFFRICATES — africadas
    # 'tʃ' es el sonido de 'ch' (chcha = comer)
    'affricates': {
        'tʃ': 'voiceless postalveolar affricate',
    },

    # NASALS — nasales
    'nasals': {
        'm': 'bilabial nasal',
        'n': 'alveolar nasal',
    },

    # LIQUIDS — líquidas
    'liquids': {
        'l': 'alveolar lateral',
        'r': 'alveolar trill',
    },

    # GLIDES — semiconsonantes
    'glides': {
        'w': 'labial-velar glide',
        'j': 'palatal glide',
    },
}

# ── Contar fonemas ────────────────────────────────────────
# Creamos una lista plana con todos los fonemas
# El bucle 'for' recorre cada categoría
# y añade sus fonemas a la lista ALL_PHONEMES

ALL_PHONEMES = []
for category in PHONEME_INVENTORY.values():
    for phoneme in category.keys():
        ALL_PHONEMES.append(phoneme)

# ── Mostrar resultado ─────────────────────────────────────
print(f"Total phonemes in Tarifit: {len(ALL_PHONEMES)}")
print()
for category, phonemes in PHONEME_INVENTORY.items():
    symbols = "  ".join(phonemes.keys())
    print(f"{category.upper():<14}: {symbols}")

# ---

# ### Deberías ver:
# ```
# Total phonemes in Tarifit: 27
#
# VOWELS        : a  i  u
# STOPS         : b  d  dˁ  g  k  q  t  tˁ
# FRICATIVES    : f  s  sˁ  z  zˁ  ʃ  x  ɣ  h  ħ  ʕ
# AFFRICATES    : tʃ
# NASALS        : m  n
# LIQUIDS       : l  r
# GLIDES        : w  j
# ```

In [ ]:
# Una tupla tiene paréntesis () y comas entre valores
frase = ('001', 'ur ffigh ara', '/ur fːiɣ ara/', 'negation', 'I did not go out')
#         id      latin          ipa              tipo         traducción

In [ ]:
datos = [
    ('001', 'ur ffigh ara', '/ur fːiɣ ara/', 'negation', 'I did not go out'),
    ('002', 'arba chcha axxam', '/arbʕa tʃːa axam/', 'statement', 'The boy ate the house'),
]

# ---

# ### Crea una celda **Text**:
# ```
# ## STEP 3 — Building the Corpus (first 10 sentences)
# ```

In [ ]:
# ============================================================
# STEP 3 — Corpus sentences
#
# Cada tupla tiene 5 elementos:
# [0] id          → número de frase  '001'
# [1] sentence    → texto en Tarifit  'ur ffigh ara'
# [2] ipa         → transcripción IPA '/ur fːiɣ ara/'
# [3] sent_type   → tipo de frase     'negation'
# [4] gloss_en    → traducción inglés 'I did not go out'
#
# Seleccionamos frases que cubran fonemas DISTINTOS
# para maximizar la cobertura fonémica
# ============================================================

CORPUS = [

    # ── NEGATION ─────────────────────────────────────────
    # Cubre: /ur/ /fː/ /iɣ/ /ara/
    # Fonemas nuevos: u r f i ɣ a
    ('001',
     'ur ffigh ara',
     '/ur fːiɣ ara/',
     'negation',
     'I did not go out'),

    # Cubre además: /ul/ /axːam/ /nːi/
    # Fonemas nuevos: l x k n
    ('002',
     'ul igh ara d axxam nni',
     '/ul iɣ ara d axːam nːi/',
     'negation+deixis',
     'Do not go toward that house'),

    # ── SVO SENTENCES ────────────────────────────────────
    # Cubre: /arba/ /tʃː/ /a/
    # Fonemas nuevos: b tʃ
    ('003',
     'arba chcha axxam',
     '/arba tʃːa axːam/',
     'SVO',
     'The boy ate at home'),

    # Cubre: /tamɣart/ /tɛzra/ /amuqːran/
    # Fonemas nuevos: m ɣ t z q
    ('004',
     'tamghart tezra amuqqran',
     '/tamɣart tɛzra amuqːran/',
     'SVO',
     'The woman sees the big one'),

    # Cubre: /wali/ /nɛtːa/ /jiwɛn/
    # Fonemas nuevos: w j
    ('005',
     'wali as netta yiwen',
     '/wali as nɛtːa jiwɛn/',
     'SVO+clitic',
     'See him, he is alone'),

    # ── LOCATIVE ─────────────────────────────────────────
    # Cubre: /tafunast/ /tɛlːa/ /ɣur/ /uxːam/
    # Fonemas nuevos: f s
    ('006',
     'tafunast tella ghur uxxam',
     '/tafunast tɛlːa ɣur uxːam/',
     'locative',
     'The cow is at the house'),

    # ── IMPERATIVE ───────────────────────────────────────
    # Cubre: /kːɛr/ /fːuɣ/ /tadːart/
    # Fonemas nuevos: k d
    ('007',
     'kker ffugh s taddart',
     '/kːɛr fːuɣ s tadːart/',
     'imperative',
     'Get up and go out to the village'),

    # ── CAUSATIVE ────────────────────────────────────────
    # Cubre: /sːkrs/ → prefijo causativo ss-
    # Fonemas nuevos: sː (geminada)
    ('008',
     'sskrs as tafunast s aman',
     '/sːkrs as tafunast s aman/',
     'causative',
     'Make the cow go down to the water'),

    # ── EMPHATIC CONSONANTS ───────────────────────────────
    # Críticas para TTS — muy específicas del bereber
    # Cubre: /aðˁu/ /jɛqːur/ /jiðˁ/
    # Fonemas nuevos: dˁ (enfática)
    ('009',
     'adhu yeqqur s yidh',
     '/aðˁu jɛqːur s jiðˁ/',
     'emphatic',
     'The wind is strong at night'),

    # ── DECLARATIVE ──────────────────────────────────────
    # Frase completa — buena para prosodia natural
    # Cubre: /tamazjɣt/ /tutlajt/ /jimazjɣɛn/
    # Fonemas nuevos: combinaciones complejas
    ('010',
     'tamazight d tutlayt n yimazighen',
     '/tamazjɣt d tutlajt n jimazjɣɛn/',
     'declarative',
     'Tamazight is the language of the Amazigh people'),
]

# ── Mostrar las frases ────────────────────────────────────
print(f"Sentences in corpus: {len(CORPUS)}")
print()
print(f"{'ID':<5} {'SENTENCE':<35} {'TYPE':<20} {'ENGLISH'}")
print("-" * 85)
for frase in CORPUS:
    id_, sent, ipa, tipo, gloss = frase
    print(f"{id_:<5} {sent:<35} {tipo:<20} {gloss}")

# ---

# ### Deberías ver:
# ```
# Sentences in corpus: 10
#
# ID    SENTENCE                            TYPE                 ENGLISH
# -------------------------------------------------------------------------------------
# 001   ur ffigh ara                        negation             I did not go out
# 002   ul igh ara d axxam nni              negation+deixis      Do not go toward...
# 003   arba chcha axxam                    SVO                  The boy ate at home
# ...
# ```

In [ ]:
# Una lista de tuplas es esto:
CORPUS = [
    ('001', 'ur ffigh ara', ...),
    ('002', 'arba chcha axxam', ...),
]

# Un DataFrame es lo mismo pero en TABLA:
#
#  id  | sentence          | ipa           | type     | gloss_en
# -----+-------------------+---------------+----------+---------
#  001 | ur ffigh ara      | /ur fːiɣ ara/ | negation | I did...
#  002 | arba chcha axxam  | /arba.../     | SVO      | The boy...

In [ ]:
CORPUS = [
    ('001',
     'ur ffigh ara',
     '/ur fːiɣ ara/',
     'negation',
     'I did not go out'),

    ('002',
     'ul igh ara d axxam nni',
     '/ul iɣ ara d axːam nːi/',
     'negation+deixis',
     'Do not go toward that house'),

    ('003',
     'arba chcha axxam',
     '/arba tʃːa axːam/',
     'SVO',
     'The boy ate at home'),

    ('004',
     'tamghart tezra amuqqran',
     '/tamɣart tɛzra amuqːran/',
     'SVO',
     'The woman sees the big one'),

    ('005',
     'wali as netta yiwen',
     '/wali as nɛtːa jiwɛn/',
     'SVO+clitic',
     'See him, he is alone'),

    ('006',
     'tafunast tella ghur uxxam',
     '/tafunast tɛlːa ɣur uxːam/',
     'locative',
     'The cow is at the house'),

    ('007',
     'kker ffugh s taddart',
     '/kːɛr fːuɣ s tadːart/',
     'imperative',
     'Get up and go out to the village'),

    ('008',
     'sskrs as tafunast s aman',
     '/sːkrs as tafunast s aman/',
     'causative',
     'Make the cow go down to the water'),

    ('009',
     'adhu yeqqur s yidh',
     '/aðˁu jɛqːur s jiðˁ/',
     'emphatic',
     'The wind is strong at night'),

    ('010',
     'tamazight d tutlayt n yimazighen',
     '/tamazjɣt d tutlajt n jimazjɣɛn/',
     'declarative',
     'Tamazight is the language of the Amazigh people'),
]

In [ ]:
for frase in CORPUS:
    print(f"ID: {frase[0]} → {len(frase)} elementos")

In [ ]:
COLUMNS = ['id', 'sentence', 'ipa', 'type', 'gloss_en']

df = pd.DataFrame(CORPUS, columns=COLUMNS)

print(f"DataFrame created: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print(df.to_string(index=False))

In [ ]:
# Dimensiones
print(f"Shape: {df.shape}")
print(f"  Rows    : {df.shape[0]}")
print(f"  Columns : {df.shape[1]}")
print()

# Nombres de columnas
print(f"Columns: {list(df.columns)}")
print()

# Contar frases por tipo
print("Sentences by type:")
print(df['type'].value_counts())
print()

# Ver solo la columna de frases
print("All sentences:")
for sentence in df['sentence']:
    print(f"  → {sentence}")

In [ ]:
CORPUS.extend([

    ('011',
     'nekkni nruh s tmazight',
     '/nɛkːni nruħ s tmaziɣt/',
     'motion+1PL',
     'We go toward Amazigh'),

    ('012',
     'ffugh d axxam ghur baba',
     '/fːuɣ d axːam ɣur baba/',
     'motion',
     'I went out toward father s house'),

    ('013',
     'amzyan yedda d baba',
     '/amzjan jɛdːa d baba/',
     'motion',
     'The young one walked with father'),

    ('014',
     'afus inu yelha',
     '/afus inu jɛlħa/',
     'possessive',
     'My hand is beautiful'),

    ('015',
     'azar n taddart yelha',
     '/azar n tadːart jɛlħa/',
     'genitive',
     'The root of the village is beautiful'),

    ('016',
     'tafunast n baba tezwa',
     '/tafunast n baba tɛzwa/',
     'possessive',
     'Father s cow ran'),

    ('017',
     'tafat n wass yellha',
     '/tafat n wasː jɛlːħa/',
     'possessive',
     'The light of the day is beautiful'),

    ('018',
     'ttwali d tafat ghef ufella',
     '/tːwali d tafat ɣɛf ufɛlːa/',
     'passive',
     'The light was seen above'),

    ('019',
     'ttwakkal gh rebbi kullu ass',
     '/tːwakːal ɣ rɛbːi kulːu asː/',
     'passive+habitual',
     'Trust in God every day'),

    ('020',
     'azul fellawen kullu',
     '/azul fɛlːawɛn kulːu/',
     'greeting',
     'Hello to all of you'),
])

print(f"Corpus size now: {len(CORPUS)} sentences")
for frase in CORPUS[10:20]:
    print(f"  [{frase[0]}] {frase[1]:<35} → {frase[4]}")

In [ ]:
CORPUS.extend([

    ('021',
     'tili d nettat ara yarin',
     '/tili d nɛtːat ara jarin/',
     'equative',
     'It is she who will write'),

    ('022',
     'tili d tmazight d tutlayt nnegh',
     '/tili d tmazjɣt d tutlajt nːɛɣ/',
     'equative',
     'Amazigh is our language'),

    ('023',
     'tili d lmut i ixussen',
     '/tili d lmut i ixusːɛn/',
     'equative',
     'It is death that is feared'),

    ('024',
     'tili d tmazight i yellhan',
     '/tili d tmazjɣt i jɛlːħan/',
     'equative',
     'It is Amazigh that is beautiful'),

    ('025',
     'isem inu d Muh',
     '/isɛm inu d muħ/',
     'introduction',
     'My name is Muh'),

    ('026',
     'iqqim yiwen ghur uxxam',
     '/iqːim jiwɛn ɣur uxːam/',
     'locative',
     'One person stayed at the house'),

    ('027',
     'ur as qqarigh ara isem',
     '/ur as qːariɣ ara isɛm/',
     'negation+clitic',
     'I did not tell him the name'),

    ('028',
     'sin iseggasen zzin',
     '/sin isɛgːasɛn zːin/',
     'numeral',
     'Two years have passed'),

    ('029',
     'yekker yiwen n wass',
     '/jɛkːɛr jiwɛn n wasː/',
     'temporal',
     'One day arose'),

    ('030',
     'tiqchi tekkes aberkan',
     '/tiqtʃi tɛkːɛs abɛrkan/',
     'SVO',
     'The girl removes the black one'),
])

print(f"Corpus size now: {len(CORPUS)} sentences")
for frase in CORPUS[20:30]:
    print(f"  [{frase[0]}] {frase[1]:<35} → {frase[4]}")

In [ ]:
CORPUS.extend([

    ('031',
     'tamurt n Arrif tella s ufella',
     '/tamurt n arːif tɛlːa s ufɛlːa/',
     'locative+toponym',
     'The land of the Rif is in the north'),

    ('032',
     'tezwa tafat n uzal',
     '/tɛzwa tafat n uzal/',
     'nature',
     'The light of noon shone'),

    ('033',
     'aman yelhan i waman',
     '/aman jɛlħan i waman/',
     'nature',
     'Good water for the water'),

    ('034',
     'kker a mmi ruh s lqbila',
     '/kːɛr a mːi ruħ s lqbila/',
     'imperative',
     'Get up my son go to the tribe'),

    ('035',
     'ul iyi ttaghlu d axxam',
     '/ul iyi tːaɣlu d axːam/',
     'negation+clitic',
     'Do not make me forget the house'),

    ('036',
     'nruḥ d nettat s taddart',
     '/nruħ d nɛtːat s tadːart/',
     'motion+comitative',
     'We went with her to the village'),

    ('037',
     'aghyul yeqqim s igenwan',
     '/aɣjul jɛqːim s igɛnwan/',
     'locative',
     'The donkey stayed under the sky'),

    ('038',
     'ssiwel fell-as s trifet',
     '/sːiwɛl fɛlːas s trifɛt/',
     'imperative+causative',
     'Call him with kindness'),

    ('039',
     'ul as qqarigh ara awalen',
     '/ul as qːariɣ ara awalɛn/',
     'negation+clitic',
     'I did not tell him the words'),

    ('040',
     'amuqqran yeqqur fell-as',
     '/amuqːran jɛqːur fɛlːas/',
     'uvular',
     'The big one is heavy on him'),
])

print(f"Corpus size now: {len(CORPUS)} sentences")
for frase in CORPUS[30:40]:
    print(f"  [{frase[0]}] {frase[1]:<35} → {frase[4]}")

In [ ]:
CORPUS.extend([

    ('041',
     'ighf inu yeqqur',
     '/iɣf inu jɛqːur/',
     'uvular',
     'My head is heavy'),

    ('042',
     'tafat tban s yifri',
     '/tafat tban s jifri/',
     'SVO',
     'The light appeared in the cave'),

    ('043',
     'wali as netta yiwen',
     '/wali as nɛtːa jiwɛn/',
     'SVO+clitic',
     'See him he is alone'),

    ('044',
     'mmlen fell-as s tmaziight',
     '/mːlɛn fɛlːas s tmazjɣt/',
     'reciprocal',
     'They spoke about him in Amazigh'),

    ('045',
     'arba ameqqran yekker ffugh',
     '/arba amɛqːran jɛkːɛr fːuɣ/',
     'geminate',
     'The big boy got up and went out'),

    ('046',
     'tafunast n baba tezwa eawd',
     '/tafunast n baba tɛzwa eawd/',
     'possessive+adverb',
     'Father s cow ran again'),

    ('047',
     'netta yura isem n tmurt',
     '/nɛtːa jura isɛm n tmurt/',
     'SVO',
     'He wrote the name of the land'),

    ('048',
     'ttwakkal gh rebbi ass kullu',
     '/tːwakːal ɣ rɛbːi asː kulːu/',
     'passive+habitual',
     'Trust in God each day'),

    ('049',
     'ul ttaghlu ara awal ameqqran',
     '/ul tːaɣlu ara awal amɛqːran/',
     'negative imperative',
     'Do not forget the great word'),

    ('050',
     'tamazight d tutlayt n yimazighen kullu',
     '/tamazjɣt d tutlajt n jimazjɣɛn kulːu/',
     'declarative',
     'Tamazight is the language of all the Amazigh people'),
])

print(f"Corpus size now: {len(CORPUS)} sentences")
print()
for frase in CORPUS[40:50]:
    print(f"  [{frase[0]}] {frase[1]:<35} → {frase[4]}")

In [ ]:
# ============================================================
# Reconstruir el DataFrame completo con las 50 frases
#
# Cada vez que añadimos frases con .extend()
# el DataFrame antiguo NO se actualiza solo
# Hay que reconstruirlo desde el CORPUS actualizado
# ============================================================

COLUMNS = ['id', 'sentence', 'ipa', 'type', 'gloss_en']

df = pd.DataFrame(CORPUS, columns=COLUMNS)

print(f"DataFrame updated: {df.shape[0]} rows x {df.shape[1]} columns")
print()

# Contar frases por tipo
print("Sentences by type:")
print(df['type'].value_counts().to_string())
print()

# Verificar que no hay filas vacías
print(f"Empty rows: {df.isnull().sum().sum()}")

In [ ]:
# ============================================================
# STEP 7 — Phoneme Coverage Analysis
#
# Estrategia:
#   1. Coger todas las transcripciones IPA del DataFrame
#   2. Extraer todos los símbolos que aparecen
#   3. Comparar con nuestro inventario ALL_PHONEMES
#   4. Calcular el porcentaje de cobertura
# ============================================================

# Paso 1: juntar todas las transcripciones IPA en un solo texto
# .str.cat() concatena todos los valores de una columna
all_ipa_text = ' '.join(df['ipa'].tolist())

print("Sample of IPA text:")
print(all_ipa_text[:100], "...")
print()

# Paso 2: limpiar el texto IPA
# Quitamos los símbolos que NO son fonemas:
# / (delimitadores), espacios, ː (marca de geminada)
def clean_ipa(text):
    # re.sub(patron, reemplazo, texto)
    # sustituye todo lo que NO sea letra IPA por nada
    cleaned = text.replace('/', '')
    cleaned = cleaned.replace(' ', '')
    return cleaned

ipa_cleaned = clean_ipa(all_ipa_text)
print(f"Total IPA characters in corpus: {len(ipa_cleaned)}")
print()

# Paso 3: contar frecuencia de cada símbolo
# Counter({'a': 45, 'i': 32, ...})
symbol_freq = Counter(ipa_cleaned)

print("Most common IPA symbols:")
for symbol, count in symbol_freq.most_common(10):
    bar = '█' * (count // 3)
    print(f"  {symbol:<4} {count:>3}x  {bar}")

In [ ]:
# ============================================================
# Calcular cobertura fonémica
#
# Para cada fonema del inventario comprobamos
# si aparece en el corpus IPA
# ============================================================

covered = []   # fonemas que SÍ aparecen
missing = []   # fonemas que NO aparecen

for phoneme in ALL_PHONEMES:
    # Comprobamos si el fonema aparece en el texto IPA limpio
    if phoneme in ipa_cleaned:
        covered.append(phoneme)
    else:
        missing.append(phoneme)

# Calcular porcentaje
coverage_pct = len(covered) / len(ALL_PHONEMES) * 100

# Mostrar resultado
print("=" * 45)
print("PHONEME COVERAGE REPORT")
print("=" * 45)
print(f"Total phonemes in inventory : {len(ALL_PHONEMES)}")
print(f"Phonemes covered            : {len(covered)}")
print(f"Phonemes missing            : {len(missing)}")
print(f"Coverage                    : {coverage_pct:.1f}%")
print()

# Mostrar cuáles están cubiertos
print("COVERED phonemes:")
for category, phonemes in PHONEME_INVENTORY.items():
    found = [p for p in phonemes if p in covered]
    if found:
        print(f"  {category:<14}: {' '.join(found)}")

print()

# Mostrar cuáles faltan
if missing:
    print("MISSING phonemes:")
    for phoneme in missing:
        # Buscar en qué categoría está
        for category, phonemes in PHONEME_INVENTORY.items():
            if phoneme in phonemes:
                desc = phonemes[phoneme]
                print(f"  {phoneme:<6} [{category}] {desc}")
else:
    print("All phonemes covered!")

In [ ]:
# ============================================================
# Calcular cobertura fonémica
#
# Para cada fonema del inventario comprobamos
# si aparece en el corpus IPA
# ============================================================

covered = []   # fonemas que SÍ aparecen
missing = []   # fonemas que NO aparecen

for phoneme in ALL_PHONEMES:
    # Comprobamos si el fonema aparece en el texto IPA limpio
    if phoneme in ipa_cleaned:
        covered.append(phoneme)
    else:
        missing.append(phoneme)

# Calcular porcentaje
coverage_pct = len(covered) / len(ALL_PHONEMES) * 100

# Mostrar resultado
print("=" * 45)
print("PHONEME COVERAGE REPORT")
print("=" * 45)
print(f"Total phonemes in inventory : {len(ALL_PHONEMES)}")
print(f"Phonemes covered            : {len(covered)}")
print(f"Phonemes missing            : {len(missing)}")
print(f"Coverage                    : {coverage_pct:.1f}%")
print()

# Mostrar cuáles están cubiertos
print("COVERED phonemes:")
for category, phonemes in PHONEME_INVENTORY.items():
    found = [p for p in phonemes if p in covered]
    if found:
        print(f"  {category:<14}: {' '.join(found)}")

print()

# Mostrar cuáles faltan
if missing:
    print("MISSING phonemes:")
    for phoneme in missing:
        # Buscar en qué categoría está
        for category, phonemes in PHONEME_INVENTORY.items():
            if phoneme in phonemes:
                desc = phonemes[phoneme]
                print(f"  {phoneme:<6} [{category}] {desc}")
else:
    print("All phonemes covered!")

In [ ]:
# ============================================================
# Ver exactamente qué fonemas faltan
# y cuántas frases necesitamos añadir para cubrirlos
# ============================================================

print("MISSING PHONEMES — need to add sentences:")
print("-" * 45)
for phoneme in missing:
    for category, phonemes in PHONEME_INVENTORY.items():
        if phoneme in phonemes:
            desc = phonemes[phoneme]
            print(f"  /{phoneme}/  [{category}]  {desc}")

In [ ]:
# ============================================================
# Añadir frases para cubrir los 6 fonemas que faltan
#
# /dˁ/ → adhu  (viento)     — enfática sonora
# /tˁ/ → atbir (paloma)     — enfática sorda
# /sˁ/ → asful (pájaro)     — fricativa enfática sorda
# /zˁ/ → azar  (pie/raíz)   — fricativa enfática sonora
# /h/  → ihi   (sí/OK)      — glotal sorda
# /ʕ/  → eawd  (de nuevo)   — faríngea sonora
# ============================================================

CORPUS.extend([

    # /dˁ/ — voiced emphatic dental stop
    # 'adhu' = viento — la d enfática es característica
    ('051',
     'adhu yeqqur s yidh',
     '/aðˁu jɛqːur s jiðˁ/',
     'emphatic_dˁ',
     'The wind is strong at night'),

    # /tˁ/ — voiceless emphatic dental stop
    # 'atbir' = paloma — t enfática inicial
    ('052',
     'atbir yella s ufella n uxxam',
     '/atˁbir jɛlːa s ufɛlːa n uxːam/',
     'emphatic_tˁ',
     'The pigeon is on top of the house'),

    # /sˁ/ — voiceless emphatic alveolar fricative
    # 'asful' = pájaro — s enfática inicial
    ('053',
     'asful yeffegh s tzeqqa',
     '/asˁful jɛfːɛɣ s tzɛqːa/',
     'emphatic_sˁ',
     'The bird flew out of the roof'),

    # /zˁ/ — voiced emphatic alveolar fricative
    # 'azeggagh' = rojo — z enfática
    ('054',
     'azeggagh d aberkan qqaren fell-as',
     '/azˁɛgːaɣ d abɛrkan qːarɛn fɛlːas/',
     'emphatic_zˁ',
     'Red and black they say about it'),

    # /h/ — voiceless glottal fricative
    # 'ihi' = sí / OK — h glotal clara
    ('055',
     'ihi ruḥ tura d axxam',
     '/ihi ruħ tura d axːam/',
     'glottal_h',
     'OK go now to the house'),

    # /ʕ/ — voiced pharyngeal fricative
    # 'eawd' = de nuevo — ʕ faríngea inicial
    ('056',
     'eawd yekker ffugh s taddart',
     '/ʕawd jɛkːɛr fːuɣ s tadːart/',
     'pharyngeal_ʕ',
     'Again he got up and went out to the village'),
])

print(f"Corpus size now: {len(CORPUS)} sentences")
print()
print("New sentences targeting missing phonemes:")
print("-" * 55)
for frase in CORPUS[50:]:
    print(f"  [{frase[0]}] /{frase[2]}/")
    print(f"        Target: {frase[3]}")
    print(f"        EN: {frase[4]}")
    print()

In [ ]:
# ============================================================
# Recalcular cobertura con las nuevas frases
# ============================================================

# Reconstruir DataFrame
df = pd.DataFrame(CORPUS, columns=COLUMNS)

# Recalcular IPA limpio
all_ipa_text = ' '.join(df['ipa'].tolist())
ipa_cleaned  = clean_ipa(all_ipa_text)

# Recalcular cobertura
covered = []
missing = []
for phoneme in ALL_PHONEMES:
    if phoneme in ipa_cleaned:
        covered.append(phoneme)
    else:
        missing.append(phoneme)

coverage_pct = len(covered) / len(ALL_PHONEMES) * 100

print("=" * 45)
print("UPDATED PHONEME COVERAGE REPORT")
print("=" * 45)
print(f"Total sentences             : {len(CORPUS)}")
print(f"Total phonemes in inventory : {len(ALL_PHONEMES)}")
print(f"Phonemes covered            : {len(covered)}")
print(f"Phonemes missing            : {len(missing)}")
print(f"Coverage                    : {coverage_pct:.1f}%")
print()

if missing:
    print("Still missing:")
    for phoneme in missing:
        print(f"  /{phoneme}/")
else:
    print("All phonemes covered!")

In [ ]:
# Ver qué fonema queda sin cubrir
print("Still missing:")
for phoneme in missing:
    for category, phonemes in PHONEME_INVENTORY.items():
        if phoneme in phonemes:
            desc = phonemes[phoneme]
            print(f"  /{phoneme}/  [{category}]  {desc}")

In [ ]:
# ============================================================
# Añadir frase con /dˁ/ explícito en la transcripción IPA
# Usamos 'iddaren' = piernas/vida — dˁ clara en posición media
# ============================================================

CORPUS.append((
    '057',
    'iddaren inu qquren fell-i',
    '/idˁːarɛn inu qːurɛn fɛlːi/',
    'emphatic_dˁ',
    'My legs are heavy on me'
))

# Reconstruir DataFrame
df = pd.DataFrame(CORPUS, columns=COLUMNS)

# Recalcular IPA limpio
all_ipa_text = ' '.join(df['ipa'].tolist())
ipa_cleaned  = clean_ipa(all_ipa_text)

# Recalcular cobertura
covered = []
missing = []
for phoneme in ALL_PHONEMES:
    if phoneme in ipa_cleaned:
        covered.append(phoneme)
    else:
        missing.append(phoneme)

coverage_pct = len(covered) / len(ALL_PHONEMES) * 100

print("=" * 45)
print("FINAL PHONEME COVERAGE REPORT")
print("=" * 45)
print(f"Total sentences             : {len(CORPUS)}")
print(f"Phonemes covered            : {len(covered)}")
print(f"Phonemes missing            : {len(missing)}")
print(f"Coverage                    : {coverage_pct:.1f}%")
print()
if missing:
    print("Still missing:")
    for phoneme in missing:
        print(f"  /{phoneme}/")
else:
    print("All phonemes covered!")

In [ ]:
# Una frase → .append()
CORPUS.append(('057', ...))

# Varias frases → .extend()
CORPUS.extend([('057', ...), ('058', ...)])

In [ ]:
# ============================================================
# STEP 8 — Visualize the corpus with matplotlib
#
# fig, axes = plt.subplots(1, 2, ...)
#   1    → número de filas de gráficas
#   2    → número de columnas de gráficas
#   figsize=(14, 5) → ancho x alto en pulgadas
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fig.suptitle(
    'Tarifit PBC Corpus — Analysis',
    fontsize=16,
    fontweight='bold'
)

# ── GRÁFICA 1: Tipos de frases (barras horizontales) ─────
type_counts = df['type'].value_counts()

# Colores — uno por cada tipo de frase
colors = [
    '#3B82F6', '#10B981', '#F59E0B', '#EF4444',
    '#8B5CF6', '#06B6D4', '#F97316', '#84CC16',
    '#EC4899', '#14B8A6', '#6366F1', '#F43F5E',
]

axes[0].barh(
    type_counts.index,    # eje Y — tipos de frases
    type_counts.values,   # eje X — cantidad
    color=colors[:len(type_counts)]
)

axes[0].set_title('Sentence Types Distribution', fontsize=12)
axes[0].set_xlabel('Number of sentences')
axes[0].set_ylabel('Type')

# Añadir número al final de cada barra
for i, v in enumerate(type_counts.values):
    axes[0].text(
        v + 0.1,   # posición X — justo después de la barra
        i,         # posición Y — altura de la barra
        str(v),    # texto — el número
        va='center',
        fontsize=9
    )

# ── GRÁFICA 2: Cobertura fonémica (circular) ─────────────
axes[1].pie(
    [len(covered), len(missing)],          # datos
    labels=[
        f'Covered ({len(covered)})',
        f'Missing ({len(missing)})'
    ],
    colors=['#82E0AA', '#F1948A'],          # verde y rojo
    autopct='%1.1f%%',                     # mostrar porcentaje
    startangle=90,                         # empezar desde arriba
    textprops={'fontsize': 11}
)

axes[1].set_title('Phoneme Inventory Coverage', fontsize=12)

# ── Mostrar ───────────────────────────────────────────────
plt.tight_layout()
plt.savefig('tarifit_pbc_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved as tarifit_pbc_analysis.png")

In [ ]:
# ============================================================
# STEP 9 — Export corpus in TTS-ready formats
#
# df.to_csv() guarda el DataFrame como archivo CSV
#   index=False  → no guardar el número de fila
#   encoding     → utf-8 para soportar símbolos IPA
# ============================================================

# ── Formato 1: CSV completo para Hugging Face ─────────────
df.to_csv('tarifit_pbc_corpus.csv', index=False, encoding='utf-8')
print("Saved: tarifit_pbc_corpus.csv")

# ── Formato 2: metadata.csv para Coqui TTS / VITS ────────
# Formato estándar: file_id|sentence|speaker
# Este es el formato que usan los sistemas TTS modernos
with open('metadata.csv', 'w', encoding='utf-8') as f:
    for _, row in df.iterrows():
        # _ ignora el índice de la fila
        # row contiene todos los campos de esa fila
        linea = f"{row['id']}|{row['sentence']}|tarifit_speaker_01\n"
        f.write(linea)
print("Saved: metadata.csv  (Coqui TTS format)")

# ── Formato 3: script de grabación ───────────────────────
# Para que un hablante nativo grabe las frases
# Incluye la transcripción IPA como guía de pronunciación
with open('recording_script.txt', 'w', encoding='utf-8') as f:
    f.write('TARIFIT PBC CORPUS — RECORDING SCRIPT\n')
    f.write('=' * 50 + '\n')
    f.write('Instructions:\n')
    f.write('- Record in a quiet room\n')
    f.write('- Natural speaking pace\n')
    f.write('- WAV format, 22050 Hz, mono\n')
    f.write('- File name: {id}.wav  (e.g. 001.wav)\n')
    f.write('=' * 50 + '\n\n')

    for _, row in df.iterrows():
        f.write(f"[{row['id']}]  {row['sentence']}\n")
        f.write(f"       IPA : {row['ipa']}\n")
        f.write(f"       EN  : {row['gloss_en']}\n\n")

print("Saved: recording_script.txt")
print()
print("All files exported successfully.")

In [ ]:
# ============================================================
# Verificar que los archivos se guardaron correctamente
# ============================================================

import os

files = [
    'tarifit_pbc_corpus.csv',
    'metadata.csv',
    'recording_script.txt',
    'tarifit_pbc_analysis.png',
]

print("Files generated:")
print("-" * 40)
for file in files:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f"  ✓ {file:<35} {size:>6} bytes")
    else:
        print(f"  ✗ {file:<35} NOT FOUND")


In [ ]:
# ============================================================
# Descargar archivos desde Google Colab
# files.download() abre el diálogo de descarga
# ============================================================

from google.colab import files

files.download('tarifit_pbc_corpus.csv')
files.download('metadata.csv')
files.download('recording_script.txt')
files.download('tarifit_pbc_analysis.png')

print("Downloads started.")

In [ ]:
# ============================================================
# Descargar archivos desde Google Colab
# files.download() abre el diálogo de descarga
# ============================================================

from google.colab import files

files.download('tarifit_pbc_corpus.csv')
files.download('metadata.csv')
files.download('recording_script.txt')
files.download('tarifit_pbc_analysis.png')

print("Downloads started.")


In [ ]:
from google.colab import files
files.download('metadata.csv')

In [ ]:
# ============================================================
# Descargar archivos desde Google Colab
# files.download() abre el diálogo de descarga
# ============================================================

from google.colab import files

files.download('tarifit_pbc_corpus.csv')
files.download('metadata.csv')
files.download('recording_script.txt')
files.download('tarifit_pbc_analysis.png')

print("Downloads started.")
